# PIO — Adu Arsitektur: bisakah model lain mengalahkan **YOLOv10n/s/m**?

Lanjutan dari notebook replikasi PIO. Di sini kita **membandingkan beberapa arsitektur deteksi pada dataset & epoch yang sama** untuk melihat mana yang mengungguli YOLOv10 yang dipakai paper.

## Rekomendasi & alasan
| Arsitektur | Tahun | Kenapa berpeluang mengalahkan YOLOv10 |
|---|---|---|
| **YOLO11** (n/s/m) | 2024 | Penerus langsung YOLOv10 (Ultralytics). Backbone **C3k2** + blok atensi **C2PSA** → mAP lebih tinggi pada ukuran setara, parameter lebih sedikit. **Paling andal** menang pada epoch sama. |
| **YOLOv12** (n/s/m) | 2025 | Berbasis **atensi** (area-attention, R-ELAN). Sering mengungguli YOLO11 pada mAP; cocok untuk scene **padat/oklusi** seperti kandang ayam. Lebih berat/lambat. |
| **YOLOv9** (t/s/m) | 2024 | **PGI + GELAN** mengurangi kehilangan informasi → kuat pada objek kecil. Pesaing tangguh. |
| **YOLOv8** (n/s/m) | 2023 | Baseline matang & stabil; pembanding "generasi sebelumnya". |
| **YOLOv10** (n/s/m) | 2024 | **Baseline paper** (NMS-free, dual assignment). Dilatih ulang di sini agar adil (epoch sama). |

> **Adu adil.** Untuk tiap *tier* (n/s/m), **semua** arsitektur memakai resep **identik dari Tabel 7 paper**: n/s → 640px, lr0 0.002, batch 4; m → 960px, lr0 0.02, batch 2; AdamW, momentum 0.9, **epoch sama**, seed sama. Tidak ada tuning per-model → perbedaan murni dari arsitektur.

> **RT-DETR (transformer)** sengaja tidak dimasukkan default: konvergensinya lambat, pada epoch sedikit hampir pasti kalah. Bisa ditambah bila mau latih lama.

> ⚠️ **Waktu.** Default = 5 keluarga × 3 tier = **15 training**. Pada Colab T4 ini lama walau hanya 3 epoch. **Pangkas `FAMILIES`/`TIERS`** atau jalankan bertahap. Untuk angka "sebenarnya", naikkan epoch (mahal).


## 0 · Setup (sama seperti notebook replikasi, Ultralytics 8.3.152 utk keadilan)

In [31]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "Aktifkan GPU"
%pip install -q "ultralytics==8.3.152" requests tqdm rarfile pandas
!apt-get -qq install -y unrar >/dev/null 2>&1 || true
!apt-get -qq install -y p7zip-full unar >/dev/null 2>&1 || true
import os, random, glob, re, collections, shutil, subprocess, zipfile
import numpy as np, pandas as pd
import ultralytics, torch
ultralytics.checks()
SEED=0
os.environ["PYTHONHASHSEED"]=str(SEED); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("ultralytics", ultralytics.__version__, "| CUDA", torch.cuda.is_available())


Ultralytics 8.3.152  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
Setup complete  (32 CPUs, 63.8 GB RAM, 365.2/3906.2 GB disk)
ultralytics 8.3.152 | CUDA True


## 1 · Data (pakai ulang dataset PIO di Drive-mu)

Sama seperti notebook utama: mount Drive, lewati unduh bila `data.rar` sudah ada, ekstrak ke disk lokal, lalu bangun split + `dataset.yaml`. Jika kamu sudah menjalankan notebook utama, `data.rar` sudah di `MyDrive/PIO/downloads` → bagian ini cepat.

In [32]:
from pathlib import Path
import os

# Folder tempat notebook berada
BASE = Path.cwd()

# Ganti bila folder hasil ekstrakmu berada di lokasi lain
DATA_ROOT = BASE / "downloads" / "data"

# Folder output training, CSV, grafik, dan bobot model
RUNS_DIR = BASE / "runs_compare"

# Tidak dipakai untuk download, tapi dibiarkan supaya kompatibel dengan cell lain
DL_DIR = BASE / "downloads"

for folder in [DL_DIR, RUNS_DIR, DATA_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root :", BASE)
print("Dataset     :", DATA_ROOT)
print("Output run  :", RUNS_DIR)

Project root : e:\CCTV\PIO
Dataset     : e:\CCTV\PIO\downloads\data
Output run  : e:\CCTV\PIO\runs_compare


In [33]:
import glob

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

images = [
    p for p in glob.glob(
        os.path.join(str(DATA_ROOT), "**", "*"),
        recursive=True
    )
    if p.lower().endswith(IMG_EXT)
]

labels = glob.glob(
    os.path.join(str(DATA_ROOT), "**", "*.txt"),
    recursive=True
)

print("Jumlah gambar :", len(images))
print("Jumlah label  :", len(labels))

if len(images) == 0:
    raise FileNotFoundError(
        f"Tidak menemukan gambar di folder dataset: {DATA_ROOT}"
    )

if len(labels) == 0:
    raise FileNotFoundError(
        f"Tidak menemukan file label .txt di folder dataset: {DATA_ROOT}"
    )

print("✓ Dataset lokal terdeteksi dan siap diproses.")

Jumlah gambar : 1487
Jumlah label  : 1489
✓ Dataset lokal terdeteksi dan siap diproses.


In [34]:
# Bangun split (pakai split bawaan data.rar bila ada) + dataset.yaml + struktur symlink lokal
import yaml
IMG_EXT=(".jpg",".jpeg",".png",".bmp")
def find_split_dirs(root):
    f={}
    for s in ("train","val","valid"):
        for d in glob.glob(os.path.join(root,"**","images",s),recursive=True): f.setdefault(("images","val" if s.startswith("val") else "train"),d)
        for d in glob.glob(os.path.join(root,"**","labels",s),recursive=True): f.setdefault(("labels","val" if s.startswith("val") else "train"),d)
    return f
def all_images(root): return sorted(p for p in glob.glob(os.path.join(root,"**","*"),recursive=True) if p.lower().endswith(IMG_EXT))
def label_for(img):
    stem=os.path.splitext(os.path.basename(img))[0]
    cand=os.path.splitext(img.replace(os.sep+"images"+os.sep,os.sep+"labels"+os.sep))[0]+".txt"
    if os.path.exists(cand): return cand
    h=glob.glob(os.path.join(DATA_ROOT,"**",stem+".txt"),recursive=True); return h[0] if h else None
def parse_house(name):
    b=os.path.basename(name); m=re.search(r"(?i)(?:^|[_\-])([CP])-?W[-_ ]?[1-6]",b)
    return {"c":"Commercial","p":"Prototype"}[m.group(1).lower()] if m else "Unknown"
def parse_week(name):
    m=re.search(r"(?i)[-_ ]W[-_ ]?([1-6])\b",os.path.basename(name)); return f"W{m.group(1)}" if m else "W?"

splits=find_split_dirs(DATA_ROOT)
PREMADE=("images","train") in splits and ("images","val") in splits
recs=[]
if PREMADE:
    for sp in ("train","val"):
        for img in all_images(splits[("images",sp)]): recs.append([img,label_for(img),sp])
else:
    pairs=[(i,label_for(i)) for i in all_images(DATA_ROOT)]; pairs=[(i,l) for i,l in pairs if l]
    rng=random.Random(SEED); st=collections.defaultdict(list)
    for i,l in pairs: st[(parse_house(i),parse_week(i))].append((i,l))
    for items in st.values():
        items=sorted(items); rng.shuffle(items); nval=round(len(items)*0.3)
        for idx,(i,l) in enumerate(items): recs.append([i,l,"val" if idx<nval else "train"])
df=pd.DataFrame(recs,columns=["img","lbl","split"])
print("Split:","BAWAAN" if PREMADE else "70/30 dibuat","| train",(df.split=="train").sum(),"| val",(df.split=="val").sum())

NORM = str(BASE / "_pio_yolo")
shutil.rmtree(NORM, ignore_errors=True)
for sub in ["images/train","images/val","labels/train","labels/val"]: os.makedirs(os.path.join(NORM,sub),exist_ok=True)
def link(s,d):
    if os.path.lexists(d): return
    try: os.symlink(os.path.abspath(s),d)
    except Exception: shutil.copy(s,d)
for r in df.itertuples():
    if not r.lbl: continue
    base=os.path.basename(r.img); stem=os.path.splitext(base)[0]
    link(r.img,os.path.join(NORM,"images",r.split,base)); link(r.lbl,os.path.join(NORM,"labels",r.split,stem+".txt"))
YAML_PATH=os.path.join(NORM,"dataset.yaml")
yaml.safe_dump({"path":NORM,"train":"images/train","val":"images/val","nc":1,"names":["pollo"]},open(YAML_PATH,"w"),sort_keys=False)
print("dataset.yaml siap:",YAML_PATH)


Split: BAWAAN | train 1035 | val 452
dataset.yaml siap: e:\CCTV\PIO\_pio_yolo\dataset.yaml


## 2 · Model zoo & konfigurasi adu

Edit `FAMILIES` / `TIERS` untuk memangkas. Resep per-tier diambil **persis** dari Tabel 7.

In [35]:
# Keluarga arsitektur & tier yang diadu (pangkas utk hemat waktu)
FAMILIES = ["yolo11"]   # yolov10 = baseline paper
TIERS    = ["m"]
QUICK_TEST = False                  # True=3 epoch (demo). False=epoch penuh (mahal).
EPOCHS   = 1 if QUICK_TEST else 100

# Resep PERSIS Tabel 7 — dipakai sama utk SEMUA arsitektur di tier itu (adu adil)
TIER_CFG = {
    "n": dict(imgsz=640, lr0=0.002, batch=4, optimizer="AdamW", momentum=0.9),
    "s": dict(imgsz=640, lr0=0.002, batch=4, optimizer="AdamW", momentum=0.9),
    "m": dict(imgsz=960, lr0=0.02,  batch=2, optimizer="AdamW", momentum=0.9),
}

def weight_name(family, tier):
    "Penamaan bobot Ultralytics (tak seragam antar versi)."
    if family == "yolov9":                       # v9: t/s/m (bukan n)
        return "yolov9" + {"n":"t","s":"s","m":"m"}[tier] + ".pt"
    return f"{family}{tier}.pt"                   # v8/v10 pakai 'v'; 11/12 tanpa 'v'

MODELS = []
for tier in TIERS:
    for fam in FAMILIES:
        w = weight_name(fam, tier)
        MODELS.append(dict(label=os.path.splitext(w)[0], family=fam, tier=tier, weight=w))
print(f"{len(MODELS)} model akan dilatih (epoch={EPOCHS}):")
for m in MODELS: print(f"  {m['label']:<12} tier={m['tier']}  ({m['weight']})")


1 model akan dilatih (epoch=100):
  yolo11m      tier=m  (yolo11m.pt)


## 3 · Training (semua model, resep identik per tier)

Tiap model gagal-load/gagal-latih dilewati dengan pesan (tidak menghentikan keseluruhan).

In [36]:
from ultralytics import YOLO
import time
trained = {}
for m in MODELS:
    c = TIER_CFG[m["tier"]]; runname = "cmp_" + m["label"]
    print(f"\n===== {m['label']} (tier {m['tier']}: imgsz={c['imgsz']} lr0={c['lr0']} batch={c['batch']} epochs={EPOCHS}) =====")
    try:
        model = YOLO(m["weight"])
    except Exception as e:
        print("  ! gagal load", m["weight"], "->", e, "(dilewati; mungkin butuh ultralytics lebih baru)"); continue
    try:
        t0 = time.time()
        model.train(data=YAML_PATH, epochs=EPOCHS, imgsz=c["imgsz"], batch=c["batch"],
                    optimizer=c["optimizer"], lr0=c["lr0"], momentum=c["momentum"],
                    seed=SEED, deterministic=True, project=RUNS_DIR, name=runname,
                    exist_ok=True, verbose=False)
        trained[m["label"]] = dict(model=model, run=runname, **m, minutes=(time.time()-t0)/60)
        print(f"  selesai {trained[m['label']]['minutes']:.1f} mnt")
    except Exception as e:
        print("  ! gagal latih", m["label"], "->", e)
print("\nSelesai. Model terlatih:", list(trained.keys()))



===== yolo11m (tier m: imgsz=960 lr0=0.02 batch=2 epochs=100) =====
New https://pypi.org/project/ultralytics/8.4.75 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.152  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\CCTV\PIO\_pio_yolo\dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.02, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.9, mos

train: Scanning E:\CCTV\PIO\_pio_yolo\labels\train... 1035 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1035/1035 [00:00<00:00, 1047.51it/s]


train: New cache created: E:\CCTV\PIO\_pio_yolo\labels\train.cache
val: Fast image access  (ping: 0.00.0 ms, read: 130.134.9 MB/s, size: 947.3 KB)


val: Scanning E:\CCTV\PIO\_pio_yolo\labels\val... 452 images, 0 backgrounds, 0 corrupt: 100%|██████████| 452/452 [00:00<00:00, 1092.93it/s]

val: New cache created: E:\CCTV\PIO\_pio_yolo\labels\val.cache


...

    100/100      15.2G     0.8076     0.3583     0.8894         58        960: 100%|██████████| 518/518 [00:24<00:00, 20.80it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 113/113 [00:03<00:00, 29.95it/s]

                   all        452      73859       0.96      0.888      0.934       0.76



100 epochs completed in 0.879 hours.
Optimizer stripped from e:\CCTV\PIO\runs_compare\cmp_yolo11m\weights\last.pt, 40.5MB
Optimizer stripped from e:\CCTV\PIO\runs_compare\cmp_yolo11m\weights\best.pt, 40.5MB

Validating e:\CCTV\PIO\runs_compare\cmp_yolo11m\weights\best.pt...
Ultralytics 8.3.152  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 113/113 [00:04<00:00, 25.45it/s]


                   all        452      73859      0.958      0.888      0.935      0.763
Speed: 0.2ms preprocess, 2.0ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to e:\CCTV\PIO\runs_compare\cmp_yolo11m
  selesai 53.4 mnt

Selesai. Model terlatih: ['yolo11m']


## 4 · Hasil & pemenang

Evaluasi tiap model pada **val yang sama**, lalu bandingkan. Baris **PAPER-v10** = klaim Tabel 8 (referensi, 100 epoch). Sel menandai **model terbaik per tier** (berdasarkan mAP@50-95).

In [37]:
PAPER_T8 = {
 "yolov10n": dict(precision=0.938, recall=0.84, mAP50=0.90, mAP50_95=0.671),
 "yolov10s": dict(precision=0.954, recall=0.86, mAP50=0.92, mAP50_95=0.70),
 "yolov10m": dict(precision=0.961, recall=0.88, mAP50=0.97, mAP50_95=0.76),
}
def read_losses(run):
    csv=os.path.join(RUNS_DIR,run,"results.csv")
    if not os.path.exists(csv): return {}
    d=pd.read_csv(csv); d.columns=[c.strip() for c in d.columns]; last=d.iloc[-1]
    def pick(kind):
        cs=[c for c in d.columns if "val" in c.lower() and kind in c.lower()]
        for key in ("loss","om","oo"):
            for c in cs:
                if key in c.lower(): return float(last[c])
        return float(last[cs[0]]) if cs else float("nan")
    return dict(box_loss=pick("box"),cls_loss=pick("cls"),dfl_loss=pick("dfl"))

rows=[]
for label,info in trained.items():
    best=os.path.join(RUNS_DIR,info["run"],"weights","best.pt")
    mdl=YOLO(best) if os.path.exists(best) else info["model"]
    r=mdl.val(data=YAML_PATH,imgsz=TIER_CFG[info["tier"]]["imgsz"],split="val",verbose=False)
    rows.append(dict(model=label,tier=info["tier"],family=info["family"],
                     precision=round(float(r.box.mp),3),recall=round(float(r.box.mr),3),
                     mAP50=round(float(r.box.map50),3),mAP50_95=round(float(r.box.map),3),
                     **{k:round(v,3) for k,v in read_losses(info["run"]).items()}))
res=pd.DataFrame(rows)
if res.empty:
    print("Belum ada model terlatih.")
else:
    res=res.sort_values(["tier","mAP50_95"],ascending=[True,False]).reset_index(drop=True)
    display(res)
    out=os.path.join(RUNS_DIR,"comparison_results.csv"); res.to_csv(out,index=False); print("Disimpan ke Drive:",out)


Ultralytics 8.3.152  Python-3.11.9 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 3117.7879.6 MB/s, size: 1034.3 KB)


val: Scanning E:\CCTV\PIO\_pio_yolo\labels\val.cache... 452 images, 0 backgrounds, 0 corrupt: 100%|██████████| 452/452 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 29/29 [00:07<00:00,  4.03it/s]


                   all        452      73859      0.958      0.888      0.935      0.766
Speed: 0.2ms preprocess, 3.0ms inference, 0.0ms loss, 7.0ms postprocess per image
Results saved to runs\detect\val7


,model,tier,family,precision,recall,mAP50,mAP50_95,box_loss,cls_loss,dfl_loss
0,yolo11m,m,yolo11,0.958,0.888,0.935,0.766,0.785,0.351,0.918


Disimpan ke Drive: e:\CCTV\PIO\runs_compare\comparison_results.csv


In [38]:
# Pemenang per tier + selisih vs baseline YOLOv10 tier sama
if not res.empty:
    for tier in sorted(res.tier.unique()):
        sub=res[res.tier==tier].sort_values("mAP50_95",ascending=False)
        win=sub.iloc[0]
        base=sub[sub.family=="yolov10"]
        bm=float(base.iloc[0]["mAP50_95"]) if not base.empty else float("nan")
        print(f"\n### TIER {tier.upper()} — pemenang: {win['model']}  (mAP50-95={win['mAP50_95']}, mAP50={win['mAP50']})")
        if not base.empty:
            d=win['mAP50_95']-bm
            verdict="MENGALAHKAN" if d>0 else ("SERI dengan" if abs(d)<1e-9 else "kalah dari")
            print(f"    vs YOLOv10{tier} (baseline, mAP50-95={bm}): {win['model']} {verdict} baseline (Δ={d:+.3f})")
        print(sub[['model','precision','recall','mAP50','mAP50_95']].to_string(index=False))



### TIER M — pemenang: yolo11m  (mAP50-95=0.766, mAP50=0.935)
  model  precision  recall  mAP50  mAP50_95
yolo11m      0.958   0.888  0.935     0.766


In [39]:
# Grafik batang mAP50-95 per tier
import matplotlib.pyplot as plt
if not res.empty:
    tiers=sorted(res.tier.unique()); fig,axes=plt.subplots(1,len(tiers),figsize=(6*len(tiers),4.5),squeeze=False)
    for ax,tier in zip(axes[0],tiers):
        sub=res[res.tier==tier].sort_values("mAP50_95")
        colors=["#d95f02" if f=="yolov10" else "#1b9e77" for f in sub.family]
        ax.barh(sub.model,sub.mAP50_95,color=colors)
        for y,(v,p) in enumerate(zip(sub.mAP50_95,sub.mAP50)): ax.text(v,y,f" {v:.3f}",va="center",fontsize=8)
        ax.set_title(f"Tier {tier} — mAP@50-95 (oranye=YOLOv10 baseline)"); ax.set_xlabel("mAP@50-95")
    plt.tight_layout(); plt.savefig(os.path.join(RUNS_DIR,"comparison_mAP.png"),dpi=130); plt.show()


<Figure size 600x450 with 1 Axes>

## 5 · Kesimpulan

- Pemenang aktual ditentukan oleh tabel & grafik di atas (pada **epoch yang kamu pilih**).
- Ekspektasi umum pada epoch sama: **YOLO11 ≥ YOLOv10**, **YOLOv12** sering tertinggi pada scene padat (tapi paling berat), **YOLOv9** kompetitif, **YOLOv8** biasanya baseline terendah.
- Semua dilatih dengan **resep identik per tier (Tabel 7)** tanpa tuning per-model → adu murni arsitektur. Bila ingin lebih jujur lagi untuk satu arsitektur, lakukan tuning lr/optimizer khusus model itu.
- **Catatan QUICK_TEST.** Bila `QUICK_TEST=True` (3 epoch), peringkat sudah indikatif tetapi angka belum final. Untuk klaim skripsi, naikkan epoch (idealnya 100) — jalankan bertahap per tier agar muat di Colab.

Output tersimpan di Drive: `runs_compare/comparison_results.csv` & `comparison_mAP.png`.


In [40]:
# ===== Model baru (Drive) vs KLAIM PAPER Tabel 8 =====
import os, glob, pandas as pd
RUNS_DIR = str(BASE / "runs_compare")
RUNS_DIR = "/content/drive/MyDrive/PIO/runs_compare"   # ganti bila beda

# Klaim paper (Tabel 8) — 100 epoch
PAPER_T8 = {
 "yolov10n": dict(box_loss=2.1, cls_loss=0.9, dfl_loss=1.95, precision=0.938, recall=0.84, mAP50=0.90, mAP50_95=0.671),
 "yolov10s": dict(box_loss=1.9, cls_loss=0.8, dfl_loss=1.90, precision=0.954, recall=0.86, mAP50=0.92, mAP50_95=0.70),
 "yolov10m": dict(box_loss=1.6, cls_loss=0.6, dfl_loss=1.75, precision=0.961, recall=0.88, mAP50=0.97, mAP50_95=0.76),
}
METRICS = ["box_loss","cls_loss","dfl_loss","precision","recall","mAP50","mAP50_95"]

def read_results_row(rd):
    csv=os.path.join(rd,"results.csv")
    if not os.path.exists(csv): return None
    d=pd.read_csv(csv); d.columns=[c.strip() for c in d.columns]
    mcol=[c for c in d.columns if ("map50-95" in c.lower()) or ("map50_95" in c.lower())]
    row=d.loc[d[mcol[0]].idxmax()] if mcol else d.iloc[-1]
    def col(p):
        cs=[c for c in d.columns if p(c.lower())]; return float(row[cs[0]]) if cs else float("nan")
    def loss(k):
        cs=[c for c in d.columns if "val" in c.lower() and k in c.lower()]
        for key in ("loss","om","oo"):
            for c in cs:
                if key in c.lower(): return float(row[c])
        return float(row[cs[0]]) if cs else float("nan")
    return dict(box_loss=loss("box"), cls_loss=loss("cls"), dfl_loss=loss("dfl"),
                precision=col(lambda c:"precision" in c), recall=col(lambda c:"recall" in c),
                mAP50=col(lambda c:("map50" in c) and ("95" not in c)),
                mAP50_95=col(lambda c:("map50-95" in c) or ("map50_95" in c)))

# baca reproduksi dari Drive
repro={}
for rd in sorted(glob.glob(os.path.join(RUNS_DIR,"cmp_*"))):
    r=read_results_row(rd)
    if r: repro[os.path.basename(rd)[4:]]=r

# tabel gabungan: kolom = klaim paper + reproduksimu
cols={}
for k in ["yolov10n","yolov10s","yolov10m"]:
    cols[f"PAPER {k}"]=[round(PAPER_T8[k][m],3) for m in METRICS]
for label in sorted(repro):
    cols[f"repro {label}"]=[round(repro[label].get(m,float('nan')),3) for m in METRICS]
table=pd.DataFrame(cols, index=METRICS)
print("=== Reproduksi (3 epoch) vs KLAIM PAPER (100 epoch) ===");
try: display(table)
except NameError: print(table.to_string())

# verdict tiap model-m: vs paper yolov10m (TAK adil: 3 vs 100 ep) & vs repro yolov10m (adil: epoch sama)
base_paper = PAPER_T8["yolov10m"]["mAP50_95"]
base_repro = repro.get("yolov10m",{}).get("mAP50_95", float("nan"))
print(f"\nBaseline mAP50-95 -> PAPER yolov10m={base_paper} (100ep) | REPRO yolov10m={round(base_repro,3) if base_repro==base_repro else 'NA'} (3ep)")
for label in sorted(repro):
    if not label.endswith("m"): continue
    v=repro[label]["mAP50_95"]
    dp=v-base_paper; dr=(v-base_repro) if base_repro==base_repro else float("nan")
    fair = "MENANG" if dr>0 else ("seri" if abs(dr)<1e-9 else "kalah")
    print(f"  {label:9} mAP50-95={v:.3f} | vs PAPER(100ep) Δ={dp:+.3f} | vs REPRO yolov10m(3ep, ADIL) Δ={dr:+.3f} -> {fair}")

=== Reproduksi (3 epoch) vs KLAIM PAPER (100 epoch) ===


,PAPER yolov10n,PAPER yolov10s,PAPER yolov10m
box_loss,2.100,1.900,1.600
cls_loss,0.900,0.800,0.600
dfl_loss,1.950,1.900,1.750
precision,0.938,0.954,0.961
recall,0.840,0.860,0.880
mAP50,0.900,0.920,0.970
mAP50_95,0.671,0.700,0.760



Baseline mAP50-95 -> PAPER yolov10m=0.76 (100ep) | REPRO yolov10m=NA (3ep)
